In [25]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    r"C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\Infinum_2026_Candidate_Dataset_Pack"
)

print("Dataset folder exists:", DATA_DIR.exists())
print("Dataset folder:", DATA_DIR)

csv_files = sorted(DATA_DIR.glob("*.csv"))

print("CSV files found:", len(csv_files))

for file in csv_files:
    print(file.name)

Dataset folder exists: True
Dataset folder: C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\Infinum_2026_Candidate_Dataset_Pack
CSV files found: 18
data_dictionary.csv
dataset_inventory.csv
episode_outcomes.csv
facility_reference.csv
followup_visits.csv
geography_reference.csv
lab_tests.csv
medicine_dispensing.csv
medicine_stock_status.csv
ncd_screening.csv
outreach_actions.csv
patient_360_reference.csv
prescriptions.csv
submission_template_action_queue.csv
submission_template_episode_predictions.csv
submission_template_linkage.csv
teleconsultations.csv
visit_history.csv


In [26]:
# STEP 2: Dataset inventory

dataset_summary = []

for file in csv_files:
    df = pd.read_csv(file)

    dataset_summary.append({
        "dataset": file.name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

dataset_summary = pd.DataFrame(dataset_summary)

dataset_summary

,dataset,rows,columns
0,data_dictionary.csv,166,4
1,dataset_inventory.csv,13,2
2,episode_outcomes.csv,5516,6
3,facility_reference.csv,27,10
4,followup_visits.csv,2575,11
5,geography_reference.csv,36,9
6,lab_tests.csv,2005,13
7,medicine_dispensing.csv,3756,14
8,medicine_stock_status.csv,1296,11
9,ncd_screening.csv,4748,18


In [27]:
# STEP 3: All columns in every dataset

for file in csv_files:
    df = pd.read_csv(file)

    print("\n" + "=" * 90)
    print(f"{file.name}")
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    print("=" * 90)

    print(df.columns.tolist())


data_dictionary.csv
Rows: 166 | Columns: 4
['file', 'field', 'description', 'competition_note']

dataset_inventory.csv
Rows: 13 | Columns: 2
['file', 'records_excluding_header']

episode_outcomes.csv
Rows: 5516 | Columns: 6
['episode_id', 'teleconsult_id', 'consult_date', 'cohort', 'lost_to_followup_label', 'dropout_stage_label']

facility_reference.csv
Rows: 27 | Columns: 10
['facility_id', 'facility_name', 'facility_type', 'district', 'block', 'latitude', 'longitude', 'network_context', 'cho_count', 'asha_linked_count']

followup_visits.csv
Rows: 2575 | Columns: 11
['visit_id', 'visit_source_patient_id', 'patient_name', 'mobile', 'village', 'visit_date', 'visit_type', 'facility_id', 'episode_id', 'clinical_status', 'referral_flag']

geography_reference.csv
Rows: 36 | Columns: 9
['village_id', 'village', 'district', 'block', 'latitude', 'longitude', 'population', 'road_access', 'mobile_connectivity']

lab_tests.csv
Rows: 2005 | Columns: 13
['lab_record_id', 'lab_source_patient_id', '

In [28]:
# STEP 4: Outcome analysis

outcomes = pd.read_csv(
    DATA_DIR / "episode_outcomes.csv"
)

print("Shape:", outcomes.shape)

print("\nColumns:")
print(outcomes.columns.tolist())

print("\nCohort distribution:")
print(outcomes["cohort"].value_counts(dropna=False))

print("\nLTFU distribution:")
print(
    outcomes["lost_to_followup_label"]
    .value_counts(dropna=False)
)

print("\nDropout stage distribution:")
print(
    outcomes["dropout_stage_label"]
    .value_counts(dropna=False)
)

Shape: (5516, 6)

Columns:
['episode_id', 'teleconsult_id', 'consult_date', 'cohort', 'lost_to_followup_label', 'dropout_stage_label']

Cohort distribution:
cohort
DEVELOPMENT    4132
EVALUATION     1384
Name: count, dtype: int64

LTFU distribution:
lost_to_followup_label
0.0    2144
1.0    1988
NaN    1384
Name: count, dtype: int64

Dropout stage distribution:
dropout_stage_label
Completed care journey    2144
NaN                       1384
Medicine not collected    1170
Review not attended        543
Test not completed         275
Name: count, dtype: int64


In [29]:
# STEP 5: Inspect patient identity fields

linkage_files = [
    "patient_360_reference.csv",
    "teleconsultations.csv",
    "ncd_screening.csv",
    "prescriptions.csv",
    "medicine_dispensing.csv",
    "lab_tests.csv",
    "followup_visits.csv",
    "visit_history.csv",
    "outreach_actions.csv"
]

for filename in linkage_files:
    df = pd.read_csv(DATA_DIR / filename)

    print("\n" + "=" * 90)
    print(filename)
    print("=" * 90)

    identity_cols = [
        col for col in df.columns
        if any(key in col.lower()
               for key in ["patient", "name", "mobile", "village"])
    ]

    for col in identity_cols:
        print(
            f"{col:35} "
            f"unique={df[col].nunique(dropna=True):5} "
            f"missing={df[col].isna().sum():5}"
        )


patient_360_reference.csv
patient_id                          unique= 5000 missing=    0
canonical_name                      unique=  798 missing=    0
masked_mobile                       unique= 3948 missing=    0
village_id                          unique=   36 missing=    0
village                             unique=   36 missing=    0

teleconsultations.csv
tele_source_patient_id              unique= 4231 missing=    0
patient_name                        unique= 1282 missing=    0
mobile                              unique= 4287 missing=  438
village                             unique=  113 missing=    0

ncd_screening.csv
ncd_source_patient_id               unique= 3589 missing=    0
patient_name                        unique= 1243 missing=    0
mobile                              unique= 3617 missing=  400
village                             unique=  113 missing=    0

prescriptions.csv
rx_source_patient_id                unique= 3449 missing=    0
patient_name                  

In [30]:
# STEP 6: Canonical Patient 360 reference

patients = pd.read_csv(
    DATA_DIR / "patient_360_reference.csv"
)

print("Shape:", patients.shape)
print("\nColumns:")
print(patients.columns.tolist())

print("\nFirst 5 records:")
display(patients.head())

Shape: (5000, 14)

Columns:
['patient_id', 'canonical_name', 'gender', 'date_of_birth', 'age_as_of_2026', 'masked_mobile', 'village_id', 'village', 'block', 'district', 'preferred_language', 'known_ncd_status', 'vulnerability_group', 'household_id']

First 5 records:


,patient_id,canonical_name,gender,date_of_birth,age_as_of_2026,masked_mobile,village_id,village,block,district,preferred_language,known_ncd_status,vulnerability_group,household_id
0,P000001,Mohan Mallaiah,M,2008-05-18,18,XXXXXX5988,VIL0035,Indresham,Patancheru,Sangareddy,Telugu,Hypertension+Diabetes,General,HH00384
1,P000002,Shankar Rao,M,1997-07-03,29,XXXXXX5269,VIL0007,Kallepally,Balanagar,Mahabubnagar,Telugu,NaN,General,HH01266
2,P000003,Mohan Basha,M,1979-08-22,47,XXXXXX9751,VIL0017,Chandur,Chandur,Nalgonda,Telugu,NaN,Low digital access,HH00709
3,P000004,Mohan Laxman,M,1981-02-20,45,XXXXXX4279,VIL0008,Bodajanampet,Balanagar,Mahabubnagar,Telugu,Hypertension,General,HH01940
4,P000005,Mohan Narsimha,M,1980-12-18,46,XXXXXX6710,VIL0014,Kondamallepally,Devarakonda,Nalgonda,Telugu,Hypertension,Elderly living alone,HH02488


In [31]:
# STEP 7: Teleconsultation structure

tele = pd.read_csv(
    DATA_DIR / "teleconsultations.csv"
)

print("Shape:", tele.shape)
print("\nColumns:")
print(tele.columns.tolist())

display(tele.head())

Shape: (5516, 23)

Columns:
['teleconsult_id', 'tele_source_patient_id', 'patient_name', 'gender', 'age', 'mobile', 'village', 'block', 'district', 'consult_date', 'facility_id', 'chief_complaint', 'diagnosis_group', 'consult_mode', 'duration_min', 'medicine_advised', 'test_advised', 'review_advised', 'review_due_days', 'distance_to_facility_km', 'preferred_language', 'connectivity_quality', 'consult_status']


,teleconsult_id,tele_source_patient_id,patient_name,gender,age,mobile,village,block,district,consult_date,...,consult_mode,duration_min,medicine_advised,test_advised,review_advised,review_due_days,distance_to_facility_km,preferred_language,connectivity_quality,consult_status
0,TC0000001,TEP0000001,Mohan Basha,M,47,9983609751,Chandur,Chandur,Nalgonda,2026-05-19,...,Audio,15,Yes,Yes,No,NaN,29.7,Telugu,Poor,Completed
1,TC0000002,TEP0000002,Mohan Laxman,M,44,9136944279,Bodajanampet,Balanagar,Mahabubnagar,2026-07-22,...,Assisted video,18,No,No,Yes,21.0,33.3,Telugu,Fair,Completed
2,TC0000003,TEP0000002,Mohan Laxman,M,47,NaN,Bodajanampet,Balanagar,Mahabubnagar,2026-08-11,...,Video,20,No,No,Yes,7.0,27.6,Telugu,Good,Completed
3,TC0000004,TEP0000003,Mohan Narsimha,M,46,9484336710,Kondamallepally,Devarakonda,Nalgonda,2026-06-02,...,Video,16,Yes,Yes,Yes,30.0,25.7,Telugu,Fair,Completed
4,TC0000005,TEP0000004,Sai Krishna,M,19,9445488147,Macharam,Jadcherla,Mahabubnagar,2026-04-24,...,Video,19,Yes,Yes,Yes,14.0,17.2,Urdu,Fair,Completed


In [32]:
# STEP 8: Episode relationship

print("Teleconsultation IDs:", tele["teleconsult_id"].nunique())
print("Outcome teleconsultation IDs:", outcomes["teleconsult_id"].nunique())

print("\nTeleconsultations without outcome record:")

missing_outcome = tele[
    ~tele["teleconsult_id"].isin(outcomes["teleconsult_id"])
]

print(len(missing_outcome))

Teleconsultation IDs: 5516
Outcome teleconsultation IDs: 5516

Teleconsultations without outcome record:
0


In [33]:
# STEP 9: Normalize identity fields for patient linkage

import re
import numpy as np

def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]", "", value)
    return value

def normalize_mobile(value):
    if pd.isna(value):
        return ""
    digits = re.sub(r"\D", "", str(value))
    return digits[-10:] if len(digits) >= 10 else digits


# Copy canonical patient table
patients_link = patients.copy()

patients_link["name_norm"] = (
    patients_link["canonical_name"]
    .apply(normalize_text)
)

patients_link["mobile_norm"] = (
    patients_link["masked_mobile"]
    .apply(normalize_mobile)
)

patients_link["village_norm"] = (
    patients_link["village"]
    .apply(normalize_text)
)


# Copy teleconsultation table
tele_link = tele.copy()

tele_link["name_norm"] = (
    tele_link["patient_name"]
    .apply(normalize_text)
)

tele_link["mobile_norm"] = (
    tele_link["mobile"]
    .apply(normalize_mobile)
)

tele_link["village_norm"] = (
    tele_link["village"]
    .apply(normalize_text)
)

print("Canonical records:", len(patients_link))
print("Teleconsultation records:", len(tele_link))

print("\nNormalized successfully.")

Canonical records: 5000
Teleconsultation records: 5516

Normalized successfully.


In [34]:
# STEP 10: Exact identity matching

# Match using normalized mobile
mobile_map = (
    patients_link[
        patients_link["mobile_norm"] != ""
    ]
    .groupby("mobile_norm")["patient_id"]
    .agg(list)
    .to_dict()
)

# Match using normalized name + village
name_village_map = (
    patients_link
    .groupby(["name_norm", "village_norm"])["patient_id"]
    .agg(list)
    .to_dict()
)


def get_mobile_candidates(row):
    if row["mobile_norm"] == "":
        return []
    return mobile_map.get(row["mobile_norm"], [])


def get_name_village_candidates(row):
    key = (row["name_norm"], row["village_norm"])
    return name_village_map.get(key, [])


tele_link["mobile_candidates"] = tele_link.apply(
    get_mobile_candidates,
    axis=1
)

tele_link["name_village_candidates"] = tele_link.apply(
    get_name_village_candidates,
    axis=1
)

print(
    "Records with unique mobile match:",
    tele_link["mobile_candidates"].apply(
        lambda x: len(x) == 1
    ).sum()
)

print(
    "Records with unique name + village match:",
    tele_link["name_village_candidates"].apply(
        lambda x: len(x) == 1
    ).sum()
)

Records with unique mobile match: 0
Records with unique name + village match: 3881


In [35]:
# STEP 11: Determine initial linkage confidence

def resolve_initial_match(row):

    mobile_candidates = row["mobile_candidates"]
    name_village_candidates = row["name_village_candidates"]

    # Strong agreement
    if (
        len(mobile_candidates) == 1
        and len(name_village_candidates) == 1
        and mobile_candidates[0] == name_village_candidates[0]
    ):
        return mobile_candidates[0], "HIGH"

    # Unique mobile match
    if len(mobile_candidates) == 1:
        return mobile_candidates[0], "MEDIUM"

    # Unique name + village match
    if len(name_village_candidates) == 1:
        return name_village_candidates[0], "MEDIUM"

    return None, "UNMATCHED"


resolved = tele_link.apply(
    resolve_initial_match,
    axis=1
)

tele_link["predicted_patient_id"] = resolved.apply(
    lambda x: x[0]
)

tele_link["linkage_confidence"] = resolved.apply(
    lambda x: x[1]
)

print(
    tele_link["linkage_confidence"]
    .value_counts(dropna=False)
)

linkage_confidence
MEDIUM       3881
UNMATCHED    1635
Name: count, dtype: int64


In [36]:
# STEP 12: Inspect unmatched records

unmatched = tele_link[
    tele_link["linkage_confidence"] == "UNMATCHED"
].copy()

print("Unmatched records:", len(unmatched))

display(
    unmatched[
        [
            "tele_source_patient_id",
            "patient_name",
            "gender",
            "age",
            "mobile",
            "village",
            "block",
            "district"
        ]
    ].head(20)
)

Unmatched records: 1635


,tele_source_patient_id,patient_name,gender,age,mobile,village,block,district
1,TEP0000002,Mohan Laxman,M,44,9136944279,Bodajanampet,Balanagar,Mahabubnagar
2,TEP0000002,Mohan Laxman,M,47,NaN,Bodajanampet,Balanagar,Mahabubnagar
5,TEP0000005,Anitha Narsimha,F,18,9474992238,Devarakonda,Devarakonda,Nalgonda
6,TEP0000005,Anitha Narsimha,F,18,9474992238,Devarakonda,Devarakonda,Nalgonda
7,TEP0000006,Ramesh Begum,M,68,9645420540,Kangti,Narayankhed,Sangareddy
8,TEP0000006,Ramesh Begum,M,64,NaN,Kangti,Narayankhed,Sangareddy
13,TEP0000008,Bhvani Mudiraj,F,61,9308761512,Patancheru,Patancheru,Sangareddy
14,TEP0000009,Sriniwas Khan,M,43,+91-94092-40836,Rajapur,Balanagar,Mahabubnagar
16,TEP0000010,Naveen Narsimha,M,18,9608842628,Nermat,Chandur,Nalgonda
20,TEP0000013,Sai Das,M,62,9299169424,Bhanu,Patancheru,Sangareddy


In [37]:
# STEP 13: Check fuzzy matching library

try:
    from rapidfuzz import fuzz, process
    print("RapidFuzz is available.")
except ImportError:
    print("RapidFuzz is NOT installed.")

RapidFuzz is available.


In [38]:
%pip install rapidfuzz

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
from rapidfuzz import fuzz, process

print("RapidFuzz installed successfully.")

RapidFuzz installed successfully.


In [40]:
# STEP 14: Fuzzy name matching for unmatched records

from rapidfuzz import fuzz, process

# Unique canonical names
canonical_names = patients_link["name_norm"].dropna().unique().tolist()

def fuzzy_name_match(name):
    if not name:
        return "", 0

    result = process.extractOne(
        name,
        canonical_names,
        scorer=fuzz.ratio
    )

    if result is None:
        return "", 0

    matched_name, score, _ = result
    return matched_name, score


unmatched[["fuzzy_name", "name_similarity"]] = (
    unmatched["name_norm"]
    .apply(lambda x: pd.Series(fuzzy_name_match(x)))
)

display(
    unmatched[
        [
            "patient_name",
            "village",
            "block",
            "district",
            "fuzzy_name",
            "name_similarity"
        ]
    ].head(20)
)

,patient_name,village,block,district,fuzzy_name,name_similarity
1,Mohan Laxman,Bodajanampet,Balanagar,Mahabubnagar,mohanlaxman,100.000000
2,Mohan Laxman,Bodajanampet,Balanagar,Mahabubnagar,mohanlaxman,100.000000
5,Anitha Narsimha,Devarakonda,Devarakonda,Nalgonda,anithanarsimha,100.000000
6,Anitha Narsimha,Devarakonda,Devarakonda,Nalgonda,anithanarsimha,100.000000
7,Ramesh Begum,Kangti,Narayankhed,Sangareddy,rameshbegum,100.000000
8,Ramesh Begum,Kangti,Narayankhed,Sangareddy,rameshbegum,100.000000
13,Bhvani Mudiraj,Patancheru,Patancheru,Sangareddy,bhavanimudiraj,96.296296
14,Sriniwas Khan,Rajapur,Balanagar,Mahabubnagar,srinivaskhan,91.666667
16,Naveen Narsimha,Nermat,Chandur,Nalgonda,naveennarsimha,100.000000
20,Sai Das,Bhanu,Patancheru,Sangareddy,saidas,100.000000


In [41]:
# STEP 15: Fuzzy similarity distribution

print(
    unmatched["name_similarity"].describe()
)

print("\nSimilarity bands:")

print(
    pd.cut(
        unmatched["name_similarity"],
        bins=[0, 60, 70, 80, 90, 100],
        include_lowest=True
    ).value_counts().sort_index()
)

count    1635.000000
mean       96.027223
std         6.378868
min        66.666667
25%        94.736842
50%       100.000000
75%       100.000000
max       100.000000
Name: name_similarity, dtype: float64

Similarity bands:
name_similarity
(-0.001, 60.0]       0
(60.0, 70.0]         4
(70.0, 80.0]        78
(80.0, 90.0]       209
(90.0, 100.0]     1344
Name: count, dtype: int64


In [42]:
# STEP 16A: Define simple normalization function

def normalize_simple(value):
    if pd.isna(value):
        return ""
    
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]", "", value)
    return value

print("normalize_simple() is ready")

normalize_simple() is ready


In [43]:
def normalize_simple(value):
    if pd.isna(value):
        return ""
    return re.sub(r"[^a-z0-9]", "", str(value).strip().lower())

In [44]:
# STEP 16: Multi-signal candidate scoring

canonical = patients_link.copy()

canonical["block_norm"] = canonical["block"].apply(normalize_simple)
canonical["district_norm"] = canonical["district"].apply(normalize_simple)

unmatched["block_norm"] = unmatched["block"].apply(normalize_simple)
unmatched["district_norm"] = unmatched["district"].apply(normalize_simple)


def score_candidate(row, candidate):
    score = 0

    # Name similarity
    name_score = fuzz.ratio(
        row["name_norm"],
        candidate["name_norm"]
    )

    if name_score >= 90:
        score += 50
    elif name_score >= 80:
        score += 35
    elif name_score >= 70:
        score += 20

    # Village
    if (
        row["village_norm"] != ""
        and row["village_norm"] == candidate["village_norm"]
    ):
        score += 20

    # Block
    if (
        row["block_norm"] != ""
        and row["block_norm"] == candidate["block_norm"]
    ):
        score += 15

    # District
    if (
        row["district_norm"] != ""
        and row["district_norm"] == candidate["district_norm"]
    ):
        score += 10

    # Gender
    if (
        "gender" in row.index
        and row["gender"] == candidate["gender"]
    ):
        score += 5

    return score

In [45]:
# STEP 16: Multi-signal candidate scoring

canonical = patients_link.copy()

canonical["block_norm"] = canonical["block"].apply(normalize_simple)
canonical["district_norm"] = canonical["district"].apply(normalize_simple)

unmatched["block_norm"] = unmatched["block"].apply(normalize_simple)
unmatched["district_norm"] = unmatched["district"].apply(normalize_simple)

print("Canonical records:", len(canonical))
print("Unmatched records:", len(unmatched))
print("Step 16 preparation completed")

display(
    unmatched[
        [
            "patient_name",
            "age",
            "gender",
            "village",
            "block",
            "district",
            "fuzzy_name",
            "name_similarity"
        ]
    ].head(10)
)

Canonical records: 5000
Unmatched records: 1635
Step 16 preparation completed


,patient_name,age,gender,village,block,district,fuzzy_name,name_similarity
1,Mohan Laxman,44,M,Bodajanampet,Balanagar,Mahabubnagar,mohanlaxman,100.000000
2,Mohan Laxman,47,M,Bodajanampet,Balanagar,Mahabubnagar,mohanlaxman,100.000000
5,Anitha Narsimha,18,F,Devarakonda,Devarakonda,Nalgonda,anithanarsimha,100.000000
6,Anitha Narsimha,18,F,Devarakonda,Devarakonda,Nalgonda,anithanarsimha,100.000000
7,Ramesh Begum,68,M,Kangti,Narayankhed,Sangareddy,rameshbegum,100.000000
8,Ramesh Begum,64,M,Kangti,Narayankhed,Sangareddy,rameshbegum,100.000000
13,Bhvani Mudiraj,61,F,Patancheru,Patancheru,Sangareddy,bhavanimudiraj,96.296296
14,Sriniwas Khan,43,M,Rajapur,Balanagar,Mahabubnagar,srinivaskhan,91.666667
16,Naveen Narsimha,18,M,Nermat,Chandur,Nalgonda,naveennarsimha,100.000000
20,Sai Das,62,M,Bhanu,Patancheru,Sangareddy,saidas,100.000000


In [46]:
# STEP 17: Multi-signal candidate scoring

from rapidfuzz import fuzz

def score_candidate(row, candidate):
    score = 0

    # Name similarity: maximum 50 points
    name_score = fuzz.ratio(
        row["name_norm"],
        candidate["name_norm"]
    )
    score += name_score * 0.50

    # Village: 20 points
    if row["village_norm"] and row["village_norm"] == candidate["village_norm"]:
        score += 20

    # Block: 10 points
    if row["block_norm"] and row["block_norm"] == candidate["block_norm"]:
        score += 10

    # District: 10 points
    if row["district_norm"] and row["district_norm"] == candidate["district_norm"]:
        score += 10

    # Gender: 5 points
    if str(row["gender"]).upper() == str(candidate["gender"]).upper():
        score += 5

    # Age: 5 points
    try:
        age_diff = abs(float(row["age"]) - float(candidate["age_as_of_2026"]))
        if age_diff == 0:
            score += 5
        elif age_diff <= 2:
            score += 3
        elif age_diff <= 5:
            score += 1
    except:
        pass

    return score


print("Candidate scoring function ready")

Candidate scoring function ready


In [47]:
# STEP 18: Find best canonical candidate

def find_best_candidate(row):
    # Block candidates by district first
    candidates = canonical[
        canonical["district_norm"] == row["district_norm"]
    ]

    # If no district match, use all canonical records
    if len(candidates) == 0:
        candidates = canonical

    best_score = -1
    best_patient = None

    for _, candidate in candidates.iterrows():
        score = score_candidate(row, candidate)

        if score > best_score:
            best_score = score
            best_patient = candidate["patient_id"]

    return pd.Series([best_patient, best_score])


unmatched[["predicted_patient_id", "candidate_score"]] = unmatched.apply(
    find_best_candidate,
    axis=1
)

print("Candidate matching completed")
print("Records scored:", len(unmatched))

display(
    unmatched[
        [
            "patient_name",
            "village",
            "district",
            "predicted_patient_id",
            "candidate_score"
        ]
    ].head(20)
)

KeyboardInterrupt: 

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# FIX — Rebuild linkage output with teleconsult_id

linkage_output = unmatched[
    [
        "teleconsult_id",
        "patient_name",
        "predicted_patient_id",
        "candidate_score"
    ]
].copy()

linkage_output["source_system"] = "teleconsultations"

linkage_output["confidence"] = pd.cut(
    linkage_output["candidate_score"],
    bins=[-1, 70, 85, 100],
    labels=["LOW", "MEDIUM", "HIGH"]
)

linkage_output = linkage_output[
    [
        "source_system",
        "teleconsult_id",
        "patient_name",
        "predicted_patient_id",
        "confidence",
        "candidate_score"
    ]
]

linkage_path = OUTPUT_DIR / "teleconsultation_linkage.csv"

linkage_output.to_csv(
    linkage_path,
    index=False
)

print("Fixed linkage saved:")
print(linkage_path)
print("Shape:", linkage_output.shape)
print("\nColumns:")
print(linkage_output.columns.tolist())

NameError: name 'OUTPUT_DIR' is not defined

In [ ]:
# STEP 20: Save linkage output

OUTPUT_DIR = DATA_DIR.parent / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

linkage_path = OUTPUT_DIR / "teleconsultation_linkage.csv"

linkage_output.to_csv(linkage_path, index=False)

print("Saved:", linkage_path)

Saved: C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\outputs\teleconsultation_linkage.csv
